
# Machine Learning aplicado, Notebook 6
## Riesgo de crédito en Taiwán: más allá de la regresión logística

**Curso:** *Machine Learning aplicado* · **Preparado por:** David Díaz, con asistencia de Claude (Anthropic) · **Entorno:** Google Colab

### De dónde venimos y a dónde vamos

En el notebook anterior aprendimos a predecir un sí o un no con la regresión logística, y sobre todo aprendimos
a **evaluar** un clasificador: matriz de confusión, recall y precision, el umbral como decisión, la curva ROC y
el AUC, el umbral óptimo con costos, y cuánto se mueve todo eso si cambia la muestra. Todo con 1.000 clientes
alemanes, donde una recta era suficiente y ningún adorno la mejoraba.

Hoy cambiamos de base: **30.000 clientes de tarjeta de crédito de un banco taiwanés** (Yeh y Lien, 2009), con
seis meses de historia de pagos y atrasos. Y cambiamos la pregunta: ¿hay modelos que le ganen a la logística, y
por qué? Vamos a probar tres cosas nuevas, cada una explicada desde cero: la logística con **términos
polinómicos** (donde la regularización, que en el notebook anterior no servía de nada, ahora sí importa), los
**árboles de decisión** (CART, dibujados caja por caja, y C5.0 con la librería `c50py`), y los dos ensambles de árboles que
dominan la predicción con datos tabulares, **random forest** y **gradient boosting**.

La regla del curso no cambia: cada modelo se mide en testing, con su intervalo, y antes de declarar un ganador
se comprueba que la diferencia sea más grande que el ruido. Y cuando un modelo gana, vamos a ver con nuestros
propios ojos **qué encontró** que la logística no podía ver.

### Qué vas a aprender hoy

1. Cuándo la regularización sí mejora testing: polinomios de grado 2 con y sin freno.
2. Qué es un árbol de decisión, por qué dibuja escalones y por qué se sobreajusta.
3. Qué es un ensamble: random forest (muchos árboles independientes promediados) y gradient boosting (árboles chicos que se corrigen en secuencia).
4. Cómo comparar modelos sin engañarse: validación cruzada, intervalos por bootstrap y la diferencia pareada.
5. Cómo mirar adentro del ganador: la forma no lineal que encontró, y el umbral con costos para usarlo.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings, time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report, recall_score, precision_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)



## 1. Los datos: tarjetas de crédito en Taiwán

Treinta mil clientes de tarjeta de crédito en 2005, con su límite de crédito, datos demográficos y **seis meses
de historia**: cuánto se atrasó cada mes, cuánto le facturaron y cuánto pagó. El objetivo es si dejó de pagar
al mes siguiente ($y = 1$): le pasó al 22%. Las columnas vienen como `x1`, `x2`; las renombramos para poder
leerlas. Si tienes el archivo del curso (`taiwan_credit.csv`), pega su link en `URL_TAIWAN`.

La variable que va a importar más es `atraso_sep`, el estado de pago en el mes más reciente: valores negativos
o cero significan que pagó al día (con matices que no importan hoy), 1 significa un mes de atraso, 2 dos
meses, y así hasta 8.


In [ ]:

URL_TAIWAN = ""    # <-- pega aquí el link del CSV del curso (taiwan_credit.csv) si lo tienes

def normalizar_link(url):
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

if URL_TAIWAN:
    tw = pd.read_csv(normalizar_link(URL_TAIWAN))
    X = tw.drop(columns=tw.columns[-1]).apply(pd.to_numeric)
    y = tw[tw.columns[-1]].astype(int).values
else:
    tw = fetch_openml("default-of-credit-card-clients", version=1, as_frame=True)
    X = tw.data.apply(pd.to_numeric)
    y = tw.target.astype(int).values

meses = ["sep", "ago", "jul", "jun", "may", "abr"]
X.columns = (["limite_credito", "sexo", "educacion", "estado_civil", "edad"] + ["atraso_" + m for m in meses]
             + ["factura_" + m for m in meses] + ["pago_" + m for m in meses])
print("Clientes:", len(X), "  Columnas:", X.shape[1], "  Proporción que no pagó:", round(y.mean(), 3))
X[["limite_credito", "edad", "atraso_sep", "atraso_ago", "factura_sep", "pago_sep"]].describe().round(0).T[["mean", "min", "50%", "max"]]



### El mismo ritual: separar, y escalar solo con training

70% para entrenar (21.000 clientes) y 30% para medir (9.000), estratificado. Escalamos con la media y la
desviación de training. Los árboles no necesitan escalar (deciden con umbrales, y un umbral no cambia si la
variable se multiplica por una constante), pero no les hace daño, así que para simplificar todos los modelos
reciben las mismas columnas escaladas.


In [ ]:

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
esc = StandardScaler().fit(X_tr)
A_tr, A_te = esc.transform(X_tr), esc.transform(X_te)
print("training:", len(y_tr), "clientes,", int(y_tr.sum()), "no pagaron  |  testing:", len(y_te), "clientes,", int(y_te.sum()), "no pagaron")



## 2. El kit de evaluación, resumido

Del notebook anterior nos traemos tres cosas, empaquetadas en una función para usarlas con cada modelo:

- El **AUC** en training y en testing: cómo ordena el modelo a los clientes por riesgo, y la brecha entre los
  dos, que es el termómetro de cuánto memorizó.
- El **recall y la precision de la clase 1** (no paga) con umbral 0,5, para conectar con la matriz de confusión.
- La **captura al 20%**: si el banco rechaza al 20% que el modelo considera más riesgoso, qué porcentaje de
  los que efectivamente no pagaron quedó rechazado. Es el AUC traducido a una decisión que un gerente entiende;
  al azar sería 20%.

Y una cuarta que vamos a necesitar al final: el **intervalo por bootstrap** del AUC en testing (remuestrear los
9.000 clientes con reposición, recalcular, repetir). Con 9.000 clientes el intervalo debería ser mucho más
angosto que el de German Credit, que iba de 0,76 a 0,87 con 300.


In [ ]:

def captura(y_real, riesgo, fraccion=0.20):
    cuantos = int(len(y_real) * fraccion)
    orden = np.argsort(-riesgo)
    return y_real[orden[:cuantos]].sum() / y_real.sum()

rng_bs = np.random.default_rng(0)
IDX_BS = [rng_bs.integers(0, len(y_te), len(y_te)) for _ in range(500)]   # las mismas remuestras para todos los modelos

def ic_auc(p):
    v = np.array([roc_auc_score(y_te[i], p[i]) for i in IDX_BS])
    return np.percentile(v, 2.5), np.percentile(v, 97.5)

RESULTADOS = {}
def evaluar(nombre, p_tr, p_te, segundos=None):
    pred = (p_te > 0.5).astype(int)
    lo, hi = ic_auc(p_te)
    RESULTADOS[nombre] = dict(auc_train=roc_auc_score(y_tr, p_tr), auc_test=roc_auc_score(y_te, p_te), ic_lo=lo, ic_hi=hi,
                              recall_1=recall_score(y_te, pred), precision_1=precision_score(y_te, pred), captura_20=captura(y_te, p_te), p_te=p_te, seg=segundos)
    r = RESULTADOS[nombre]
    print(f"{nombre:38s} AUC train {r['auc_train']:.3f}  test {r['auc_test']:.3f}  [{lo:.3f}, {hi:.3f}]   brecha {r['auc_train']-r['auc_test']:+.3f}   "
          f"recall {r['recall_1']:.3f}  precision {r['precision_1']:.3f}  captura20 {r['captura_20']:.3f}" + (f"   ({segundos:.0f} s)" if segundos else ""))



## 3. La logística de siempre, y la logística con polinomios

### Punto de partida

La misma regresión logística del notebook anterior, sin regularizar. Es el modelo contra el que se mide todo lo
demás.


In [ ]:

t0 = time.time()
logit = LogisticRegression(penalty=None, max_iter=5000).fit(A_tr, y_tr)
evaluar("logística lineal", logit.predict_proba(A_tr)[:, 1], logit.predict_proba(A_te)[:, 1], time.time() - t0)



### Polinomios de grado 2: donde la regularización sí importa

La logística supone que cada variable empuja el riesgo siempre en la misma dirección y al mismo ritmo, y que
las variables no interactúan. Una forma de relajar eso sin cambiar de modelo es agregar columnas: el cuadrado de
cada variable y el producto de cada par. Con 23 variables salen 299 columnas ($23$ originales, $23$ cuadrados y
$23 \cdot 22 / 2 = 253$ productos). La logística sigue siendo una recta, pero ahora en 275 dimensiones, y esa
recta puede dibujar curvas y cruces en las 23 originales.

El precio es que 299 pesos son muchos más grados de libertad, y con ellos el modelo puede usar esa libertad para
ajustarse al ruido. Lo vemos con dos versiones: la de grado 2 sin regularizar, y la de grado 2 con la $\lambda$
elegida por validación cruzada de 5 pliegues sobre training (`LogisticRegressionCV` prueba varios valores de
$C = 1/\lambda$ y se queda con el mejor sin mirar testing). Y después repetimos el experimento con solo 1.000
clientes de training, porque la pregunta de fondo es cuántos pesos puede pagar la información disponible.


In [ ]:

poly = PolynomialFeatures(degree=2, include_bias=False)
P_tr = poly.fit_transform(A_tr); P_te = poly.transform(A_te)
esc2 = StandardScaler().fit(P_tr)
P_tr, P_te = esc2.transform(P_tr), esc2.transform(P_te)
print("Columnas de grado 2:", P_tr.shape[1])

t0 = time.time()
poli_libre = LogisticRegression(penalty=None, max_iter=3000).fit(P_tr, y_tr)
evaluar("logística grado 2, sin regularizar", poli_libre.predict_proba(P_tr)[:, 1], poli_libre.predict_proba(P_te)[:, 1], time.time() - t0)

t0 = time.time()
poli_cv = LogisticRegressionCV(Cs=8, cv=5, max_iter=3000, n_jobs=-1).fit(P_tr, y_tr)
evaluar("logística grado 2, L2 por validación cruzada", poli_cv.predict_proba(P_tr)[:, 1], poli_cv.predict_proba(P_te)[:, 1], time.time() - t0)
print("C elegido por validación cruzada:", poli_cv.C_[0], " (lambda =", round(1 / poli_cv.C_[0], 4), ")")



Dos lecturas. Primero, el grado 2 le gana a la recta por 0,035 de AUC (0,750 contra 0,715), con un intervalo que
no toca el cero: en esta base **hay algo no lineal que encontrar**. Segundo, con 21.000 clientes ni siquiera 299
columnas sobreajustan mucho: la brecha sin regularizar es 0,031 y la regularización la reduce a 0,023 ganando
apenas 0,002 de AUC en testing. Es German Credit otra vez, con más datos. Entonces, ¿cuándo importa la
regularización? Cuando los pesos son muchos **respecto de las filas**. Lo comprobamos con el mismo modelo de
grado 2 entrenado con solo 1.000 clientes:


In [ ]:

P_chico, y_chico = P_tr[:1000], y_tr[:1000]
libre_chico = LogisticRegression(penalty=None, max_iter=5000).fit(P_chico, y_chico)
cv_chico = LogisticRegressionCV(Cs=8, cv=5, max_iter=5000, n_jobs=-1).fit(P_chico, y_chico)
for nombre, m in [("grado 2 con 1.000 clientes, sin regularizar", libre_chico), ("grado 2 con 1.000 clientes, L2 por validación cruzada", cv_chico)]:
    print(f"{nombre:55s} AUC train {roc_auc_score(y_chico, m.predict_proba(P_chico)[:, 1]):.3f}   test {roc_auc_score(y_te, m.predict_proba(P_te)[:, 1]):.3f}")
print("lambda elegida:", round(1 / cv_chico.C_[0], 3))



Ahí está la diferencia que en German Credit no apareció: con 1.000 filas y 299 pesos, el modelo sin freno acierta
casi todo training y se desploma en testing; el regularizado sacrifica training y generaliza mucho mejor. La
regularización es un seguro contra tener más pesos que información, y su valor depende de esa razón, no del
modelo. Con datos abundantes el seguro casi no se usa; con datos escasos es lo único que salva al modelo.

Volvamos a los 21.000 clientes. El grado 2 mejoró sobre la recta, así que hay una forma no lineal. La pregunta es
qué forma tiene, y para eso sirven los árboles.



## 4. Árboles de decisión, desde cero

Un **árbol de decisión** es una secuencia de preguntas de sí o no sobre las variables. Empieza con todos los
clientes en la raíz y busca la pregunta que mejor separa a los que pagan de los que no (por ejemplo, "¿atraso
de septiembre mayor o igual a 2?"); parte a los clientes en dos grupos según la respuesta, y en cada grupo
vuelve a buscar la mejor pregunta, hasta llegar a grupos pequeños o suficientemente puros. Cada grupo final
(**hoja**) predice la fracción de clientes que no pagó dentro de él.

Cómo elige la pregunta: prueba todas las variables y todos los cortes posibles, y se queda con el que más
reduce la **impureza** de los dos grupos resultantes (la medida habitual es el índice de Gini, $1 - \sum_k p_k^2$ con $p_k$ la fracción de cada clase
en el grupo, que con dos clases se reduce a $2p(1-p)$: vale 0 si el grupo es todo de una clase y 0,5 si está mitad y mitad). No hay pesos, no hay recta, no hay que
escalar: el árbol solo pregunta y parte.

Dos consecuencias que hay que tener claras. Un árbol dibuja **escalones**, nunca rampas: dentro de cada hoja el
riesgo es constante. Eso es una ventaja cuando la realidad tiene saltos (un umbral a partir del cual todo
cambia) y una desventaja cuando la realidad es suave. Y un árbol **profundo memoriza**: si lo dejamos crecer
hasta que cada hoja tenga dos clientes, acierta el 100% de training y poco de testing. La profundidad máxima
es el freno, y se elige igual que $\lambda$: con validación cruzada.


In [ ]:

print("Profundidad, AUC por validación cruzada en training (5 pliegues):")
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
for prof in [2, 4, 6, 8, 12, None]:
    a = cross_val_score(DecisionTreeClassifier(max_depth=prof, min_samples_leaf=20, random_state=0), A_tr, y_tr, cv=cv5, scoring="roc_auc", n_jobs=-1)
    print(f"   profundidad {str(prof):5s}  AUC {a.mean():.3f} +/- {a.std():.3f}")


In [ ]:

arbol = DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=0).fit(A_tr, y_tr)
evaluar("árbol de decisión, profundidad 4", arbol.predict_proba(A_tr)[:, 1], arbol.predict_proba(A_te)[:, 1])
arbol_hondo = DecisionTreeClassifier(max_depth=None, random_state=0).fit(A_tr, y_tr)
evaluar("árbol de decisión, sin límite", arbol_hondo.predict_proba(A_tr)[:, 1], arbol_hondo.predict_proba(A_te)[:, 1])

# el árbol de profundidad 2, entrenado sobre las columnas sin escalar para poder leer los cortes
arbol2 = DecisionTreeClassifier(max_depth=2, min_samples_leaf=20, random_state=0).fit(X_tr, y_tr)
texto = export_text(arbol2, feature_names=list(X.columns), decimals=2, show_weights=True)
print("Árbol de profundidad 2 (cortes en unidades originales):")
print(texto)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# El mismo árbol de profundidad 4, entrenado sobre las columnas sin escalar para que los cortes se lean en unidades originales.
# Un árbol no cambia si las columnas se escalan (solo pregunta "¿mayor que un umbral?"), así que el AUC es el mismo.
arbol4 = DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=0).fit(X_tr, y_tr)
print("AUC en testing, árbol de profundidad 4 sobre columnas crudas:", round(roc_auc_score(y_te, arbol4.predict_proba(X_te)[:, 1]), 3), " (escaladas:", round(RESULTADOS["árbol de decisión, profundidad 4"]["auc_test"], 3), ")")

fig, ax = plt.subplots(figsize=(12, 5))
plot_tree(arbol2, feature_names=list(X.columns), class_names=["paga", "no paga"], filled=True, proportion=True, impurity=False, rounded=True, fontsize=10, ax=ax)
ax.set_title("Árbol de profundidad 2: en cada caja, la pregunta, la fracción de clientes que llega y la fracción [paga, no paga]")
plt.show()

fig, ax = plt.subplots(figsize=(30, 11))
plot_tree(arbol4, feature_names=list(X.columns), class_names=["paga", "no paga"], filled=True, proportion=True, impurity=False, rounded=True, fontsize=7, ax=ax)
ax.set_title("Árbol de profundidad 4, mínimo 20 clientes por hoja (16 hojas como máximo)")
plt.show()

Cómo leer las cajas. Cada caja es una pregunta; a la izquierda van los clientes que responden "sí" (la condición se
cumple) y a la derecha los que responden "no". `samples` es la fracción de los 21.000 clientes de training que llega a
esa caja, y `value` es la fracción que paga y la que no paga dentro de ella; el color va de naranjo (casi todos pagan) a
azul (la mayoría no paga), y mientras más intenso, más pura la caja. Las hojas de abajo son los escalones del
modelo: cada cliente de testing cae en una sola hoja y recibe como probabilidad la fracción de "no paga" de esa hoja.

Fíjate en dos cosas. La raíz pregunta por `atraso_sep` con el corte en 1,5 (es decir, dos meses o más de atraso), y
el subárbol de la derecha, con el 10% de los clientes, concentra a los que no pagan (70% dentro de él, contra 17% a
la izquierda). Y las primeras preguntas de los dos lados son sobre los atrasos de los meses anteriores; el límite de
crédito, la edad y las facturas recién aparecen en el último nivel y para grupos chicos. La historia de pagos es la
que separa, igual que van a decir la importancia de variables y el boosting más adelante.


El árbol de profundidad 2 ya cuenta la historia: la primera pregunta es sobre el atraso de septiembre, y el
corte cae entre 1 y 2 meses de atraso. Un árbol solo, aunque bien podado, no le gana a la logística de grado 2,
y sin límite memoriza (mira la brecha). Lo que sigue es cómo convertir muchos árboles mediocres en un modelo
bueno.


### C5.0: el otro árbol clásico, con `c50py`

El árbol de scikit-learn es CART (Breiman, Friedman, Olshen y Stone, 1984): cortes binarios, impureza de Gini, y la
profundidad o el mínimo por hoja como freno. Hay otra familia igual de clásica, la de Quinlan: ID3, C4.5 y su versión
comercial **C5.0**, que es la que usan `C50` en R y muchos motores de reglas en bancos. Se diferencia de CART en tres
cosas que vale la pena conocer:

1. Elige la pregunta con la **razón de ganancia** (ganancia de información normalizada por la entropía de la
   partición) en vez de Gini, lo que castiga las variables con muchos valores distintos.
2. Trata las **variables categóricas como categóricas**: una pregunta puede abrir una rama por categoría, sin
   convertirlas a columnas 0/1. Aquí `sexo`, `educacion` y `estado_civil` vienen codificadas como enteros y a CART le
   da lo mismo; a C5.0 se le dice cuáles son categorías.
3. **Poda pesimista**: deja crecer el árbol y después lo recorta desde las hojas hacia arriba comparando el error de
   cada subárbol con una cota superior de su error (el factor de confianza `cf`, 0,25 por defecto). No hay que elegir
   una profundidad; el árbol queda tan grande como los datos justifiquen. Además tiene su propio boosting (`trials`)
   y puede convertir el árbol en reglas legibles.

La versión de Python que usamos es **`c50py`** (`pip install c50py`, código en `github.com/daviddiazsolis/c50py`),
una implementación abierta del algoritmo con la interfaz de scikit-learn (`fit`, `predict_proba`, `export_graphviz`).
El notebook *Árboles 06, C5.0* del sitio Tree & Ensemble Foundations lo recorre a fondo; aquí lo ponemos en la misma
mesa que los demás modelos. Un detalle que importa para leer el resultado: C5.0 poda mirando el **acierto**, no el
AUC, así que tiende a dejar pocas hojas con probabilidades gruesas.

In [ ]:
try:
    import c50py
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "c50py>=0.3.0"], check=True)
    import c50py
from c50py import C5Classifier
print("c50py", c50py.__version__)

CATEGORICAS = ["sexo", "educacion", "estado_civil"]
COLS = list(X.columns)
Xo_tr, Xo_te = X_tr.values.astype(object), X_te.values.astype(object)     # c50py recibe las columnas crudas (sin escalar), con las categóricas como tales

for msl in [100, 20]:
    t0 = time.time()
    c5 = C5Classifier(categorical_features=CATEGORICAS, min_samples_leaf=msl).fit(Xo_tr, y_tr, feature_names=COLS)
    evaluar(f"C5.0 (c50py), mínimo {msl} por hoja", c5.predict_proba(Xo_tr)[:, 1], c5.predict_proba(Xo_te)[:, 1], time.time() - t0)
    if msl == 100:
        c5_chico = c5

print()
print("El árbol de C5.0 con mínimo 100 por hoja, como reglas:")
c5_chico.print_tree(feature_names=COLS)
try:
    import graphviz
    display(graphviz.Source(c5_chico.export_graphviz(feature_names=COLS, class_names=["paga", "no paga"], format="dot")))
except Exception as e:
    print("(Graphviz no disponible para dibujarlo:", e, ")")

Tres lecturas. Primero, el árbol de C5.0 es **más chico y más legible** que el CART de profundidad 4: con mínimo 100 por
hoja quedan seis hojas, todas sobre los atrasos, y cada hoja se lee como una regla de negocio ("dos meses o más de
atraso en septiembre y algún atraso en julio: no paga"). Segundo, su **AUC es menor** (0,67 con 100 por hoja y 0,69 con
20, contra 0,74 del CART de profundidad 4), y no es porque el algoritmo sea peor sino porque optimiza otra cosa: la
poda pesimista se queda con el árbol que mejor **acierta**, y pocas hojas significan pocas probabilidades distintas,
es decir, muchos empates en el orden que mide el AUC. Es la diferencia entre un modelo para explicar y un modelo
para ordenar. Tercero, la brecha training contra testing es de las más chicas de la tabla: la poda pesimista cumple su
promesa de no memorizar. En el sitio de árboles se ve el caso contrario, con variables categóricas de muchas
categorías, donde C5.0 le gana a CART con dummies. Más abajo lo volvemos a ver con su boosting.


## 5. Ensambles: muchos árboles valen más que uno

Un **ensamble** combina las predicciones de muchos modelos. La intuición es la de pedirle a veinte tasadores el
precio de una casa: cada uno tiene sus manías, pero si las manías son distintas se cancelan al promediar, y el
promedio suele ser mejor que casi cualquiera de ellos. Hay dos maneras de construir el conjunto, y son ideas
distintas.

### Random forest: árboles independientes, promediados

Se entrenan cientos de árboles **profundos**, cada uno sobre una muestra distinta de training tomada al azar
con reposición (así cada árbol ve un training un poco distinto) y, en cada pregunta, mirando solo un
subconjunto al azar de las variables (así los árboles no se parecen entre sí). La predicción es el promedio de
las hojas. Cada árbol por sí solo memoriza; el promedio de muchos árboles que memorizaron cosas distintas no.
Es el modelo más difícil de arruinar que existe: casi no tiene qué afinar, y agregar árboles nunca empeora.

### Gradient boosting: árboles chicos, en secuencia, corrigiendo

Aquí los árboles no son independientes. Se parte con una predicción trivial (la tasa base), se mira en qué
clientes se equivocó y por cuánto (los **residuos**), y se entrena un árbol **chico** (profundidad 3, un
tronco) para predecir esos residuos. Se suma ese árbol a la predicción, con un factor pequeño llamado
**tasa de aprendizaje**, y se repite: cientos de veces, cada árbol corrigiendo lo que dejaron los anteriores.
Es el modelo que gana casi todas las competencias con datos tabulares (XGBoost y LightGBM son versiones rápidas
de esta idea). A diferencia del random forest, sí puede sobreajustar si se le dan demasiadas rondas, así que el
número de árboles y la tasa se eligen con validación.

Los dos se entrenan sobre las 23 columnas originales: los árboles no necesitan polinomios (ellos mismos
encuentran los umbrales e interacciones) ni escalado.


In [ ]:

t0 = time.time()
bosque = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0).fit(A_tr, y_tr)
evaluar("random forest, 300 árboles", bosque.predict_proba(A_tr)[:, 1], bosque.predict_proba(A_te)[:, 1], time.time() - t0)

t0 = time.time()
gb = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=0).fit(A_tr, y_tr)
evaluar("gradient boosting, 300 árboles", gb.predict_proba(A_tr)[:, 1], gb.predict_proba(A_te)[:, 1], time.time() - t0)


In [ ]:
# Cómo son por dentro: un árbol del bosque (profundo; se muestra truncado) y el primer árbol del boosting (chico, sobre residuos)
un_arbol = bosque.estimators_[0]
print("Primer árbol del random forest: profundidad", un_arbol.get_depth(), ", hojas", un_arbol.get_n_leaves(), " (se dibuja hasta profundidad 3)")
fig, ax = plt.subplots(figsize=(24, 8))
plot_tree(un_arbol, max_depth=3, feature_names=list(X.columns), class_names=["paga", "no paga"], filled=True, proportion=True, impurity=False, rounded=True, fontsize=8, ax=ax)
ax.set_title("Random forest: los tres primeros niveles del árbol 1 de 300 (cortes en unidades escaladas; cada '(...)' sigue creciendo)")
plt.show()

primer_gb = gb.estimators_[0, 0]
print("Primer árbol del gradient boosting: profundidad", primer_gb.get_depth(), ", hojas", primer_gb.get_n_leaves())
fig, ax = plt.subplots(figsize=(24, 7))
plot_tree(primer_gb, feature_names=list(X.columns), filled=True, impurity=False, rounded=True, fontsize=8, ax=ax)
ax.set_title("Gradient boosting: árbol 1 de 300. Predice residuos (value > 0 empuja hacia 'no paga'), y se suma con tasa 0,05")
plt.show()

Los dos dibujos muestran la diferencia de fondo entre los ensambles. El árbol del bosque es **profundo** (cientos de
hojas; aquí solo se ven tres niveles) y sus cortes cambian de un árbol a otro porque cada uno vio una muestra distinta
y un subconjunto distinto de variables: por sí solo memoriza, y el promedio de 300 que memorizaron cosas distintas es
lo que generaliza. El árbol del boosting es **chico** (profundidad 3, ocho hojas) y no predice "paga" o "no paga": predice
un **residuo**, cuánto le falta a la predicción acumulada en cada grupo de clientes, y su aporte entra multiplicado por
la tasa de aprendizaje 0,05. Fíjate en que el primer árbol del boosting parte por `atraso_sep` con el mismo corte que
el árbol simple: el primer residuo grande es justo el escalón de los dos meses de atraso.

C5.0 también tiene boosting, anterior al gradient boosting: cada ronda (`trials`) reentrena el árbol dando más peso a
los clientes que las rondas anteriores clasificaron mal (la idea de AdaBoost), y las rondas votan. Lo agregamos a la
comparación con diez rondas:

In [ ]:
t0 = time.time()
c5_boost = C5Classifier(categorical_features=CATEGORICAS, min_samples_leaf=100, trials=10, random_state=0).fit(Xo_tr, y_tr, feature_names=COLS)
evaluar("C5.0 con boosting, 10 rondas", c5_boost.predict_proba(Xo_tr)[:, 1], c5_boost.predict_proba(Xo_te)[:, 1], time.time() - t0)

Con diez rondas el C5.0 pasa de 0,67 a 0,77 de AUC en testing: el boosting convierte un árbol de seis hojas en un
modelo que ordena casi tan bien como el random forest (0,773) y se queda a una centésima del gradient boosting
(0,780), a cambio de perder la lectura como reglas y de una brecha training contra testing mayor. La tabla que sigue
los pone a todos juntos.

In [ ]:

tabla = pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk != "p_te"} for k, v in RESULTADOS.items()}).T
tabla["intervalo AUC 95%"] = tabla.apply(lambda r: f"[{r.ic_lo:.3f}, {r.ic_hi:.3f}]", axis=1)
print(tabla[["auc_train", "auc_test", "intervalo AUC 95%", "recall_1", "precision_1", "captura_20", "seg"]].round(3).to_string())



## 6. Comparar sin engañarse

La tabla ordena los modelos por AUC de testing, pero antes de declarar un ganador hay que responder dos
preguntas: ¿la diferencia es más grande que el ruido? y ¿se sostiene en otras muestras?

**Los intervalos por bootstrap** ya están en la tabla, y con 9.000 clientes son angostos, del orden de más o
menos 0,013. Pero comparar dos intervalos que se superponen un poco no es la prueba correcta, porque los dos
modelos se evalúan sobre los **mismos** clientes y sus errores están correlacionados. La prueba correcta es la
**diferencia pareada**: en cada remuestra de bootstrap se calcula el AUC de los dos modelos sobre los mismos
clientes y se resta; si el intervalo de la diferencia no contiene el cero, la diferencia es real.

**La validación cruzada** responde la segunda pregunta: entrenar y medir cinco veces sobre pliegues distintos
de training. Tarda un par de minutos con el boosting.


In [ ]:

def diferencia_pareada(a, b):
    pa, pb = RESULTADOS[a]["p_te"], RESULTADOS[b]["p_te"]
    d = np.array([roc_auc_score(y_te[i], pa[i]) - roc_auc_score(y_te[i], pb[i]) for i in IDX_BS])
    print(f"AUC({a}) - AUC({b}) = {RESULTADOS[a]['auc_test'] - RESULTADOS[b]['auc_test']:+.4f}   IC 95% pareado [{np.percentile(d, 2.5):+.4f}, {np.percentile(d, 97.5):+.4f}]")

diferencia_pareada("gradient boosting, 300 árboles", "logística lineal")
diferencia_pareada("gradient boosting, 300 árboles", "logística grado 2, L2 por validación cruzada")
diferencia_pareada("gradient boosting, 300 árboles", "random forest, 300 árboles")
diferencia_pareada("logística grado 2, L2 por validación cruzada", "logística lineal")


In [ ]:

modelos_cv = {
    "logística lineal": LogisticRegression(penalty=None, max_iter=5000),
    "random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0),
    "gradient boosting": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=0),
}
print("AUC por validación cruzada (5 pliegues sobre training):")
for nombre, m in modelos_cv.items():
    a = cross_val_score(m, A_tr, y_tr, cv=cv5, scoring="roc_auc", n_jobs=-1)
    print(f"   {nombre:20s} {a.mean():.3f} +/- {a.std():.3f}   de {a.min():.3f} a {a.max():.3f}")



Con esto la comparación queda cerrada: el intervalo pareado entre boosting y logística lineal no toca el cero, y
los rangos de validación cruzada de los dos ni se acercan. La diferencia es real. Entre boosting y random forest
es más chica, y hay que leer el intervalo pareado para decidir si es distinguible.



### Cuando el AUC empata, la curva decide

El AUC es un promedio sobre toda la curva ROC, y toda la curva incluye zonas donde nadie opera: un banco no
aprueba tarjetas marcando al 80% de los buenos clientes. Lo que importa al comparar modelos es el recall que da
cada uno **en la tasa de falsos positivos que el negocio tolera**: un emisor conservador se para en 5%; una
campaña de cobranza preventiva, donde marcar de más cuesta una llamada, puede irse a 30% o 40%. Dos modelos
con AUC casi igual pueden cruzarse, uno mejor a la izquierda y otro a la derecha, y en ese caso la pregunta
correcta no es cuál tiene más AUC sino cuál atrapa más morosos en *mi* punto.

Abajo, las curvas de los siete modelos con zoom a la zona de operación (falsos positivos hasta 30%), y una tabla
con el recall de cada uno en varias tolerancias más el **AUC parcial** (`roc_auc_score` con `max_fpr`), que resume
la curva solo hasta esa tasa. Fíjate en el árbol de profundidad 4 contra la logística de grado 2: por AUC pierde
el árbol, pero con 5% de falsos positivos atrapa más, porque su escalón grande cae justo ahí; con 50% se da vuelta.


In [ ]:

tolerancias = [0.05, 0.10, 0.20, 0.30, 0.50]
filas = []
fig = go.Figure()
for nombre, r in RESULTADOS.items():
    fpr, tpr, _ = roc_curve(y_te, r["p_te"])
    fig.add_scatter(x=fpr, y=tpr, mode="lines", name=f"{nombre} (AUC {r['auc_test']:.3f})")
    fila = {"modelo": nombre, "AUC": r["auc_test"]}
    for f in tolerancias:
        fila[f"recall con FP {int(f*100)}%"] = float(np.interp(f, fpr, tpr))
        fila[f"AUC parcial hasta {int(f*100)}%"] = roc_auc_score(y_te, r["p_te"], max_fpr=f)
    filas.append(fila)
fig.add_scatter(x=[0, 0.3], y=[0, 0.3], mode="lines", name="azar", line=dict(dash="dash", color="gray"))
fig.update_layout(title="Curvas ROC en testing, zoom a la zona de operación (FP hasta 30%)", xaxis_title="tasa de falsos positivos",
                  yaxis_title="recall", xaxis_range=[0, 0.3], yaxis_range=[0, 0.8], height=520, width=760)
fig.show()
tabla_roc = pd.DataFrame(filas).set_index("modelo")
print("Recall en cada tolerancia de falsos positivos (quién atrapa más morosos en cada punto):")
print(tabla_roc[[c for c in tabla_roc.columns if c.startswith("recall")] + ["AUC"]].round(3).sort_values("recall con FP 5%", ascending=False).to_string())
print()
print("AUC parcial (scikit-learn, max_fpr; estandarizado para que 0,5 sea el azar y 1 lo perfecto):")
print(tabla_roc[[c for c in tabla_roc.columns if c.startswith("AUC parcial")]].round(3).to_string())



Lectura: con 5% de falsos positivos el árbol de profundidad 4 empata con el boosting y supera al random forest y a
la logística de grado 2, aunque por AUC sea el cuarto; con 50% queda último de ese grupo. Random forest y boosting,
separados por 0,007 de AUC, son indistinguibles en 5% y se separan recién más a la derecha. El AUC ordena en promedio;
tu punto de operación puede ordenar distinto, y por eso conviene mirar la curva (o el AUC parcial) en la zona donde
de verdad vas a trabajar.



## 7. Qué encontró el boosting que la logística no podía ver

Esta es la parte que vale la pena entender bien. Tomemos la variable más importante, `atraso_sep`, y comparemos
para cada valor que toma tres cosas: qué porcentaje de esos clientes **realmente** no pagó, qué riesgo promedio
les asignó la logística lineal, y cuál les asignó el boosting.


In [ ]:

comp = pd.DataFrame({"atraso_sep": X_te["atraso_sep"].values, "realmente_no_pago": y_te,
                     "logistica": RESULTADOS["logística lineal"]["p_te"], "grado_2": RESULTADOS["logística grado 2, L2 por validación cruzada"]["p_te"],
                     "boosting": RESULTADOS["gradient boosting, 300 árboles"]["p_te"]})
por_atraso = comp.groupby("atraso_sep").agg(clientes=("realmente_no_pago", "size"), realmente_no_pago=("realmente_no_pago", "mean"),
                                            logistica=("logistica", "mean"), grado_2=("grado_2", "mean"), boosting=("boosting", "mean"))
print(por_atraso[por_atraso["clientes"] >= 20].round(3))



Lee la columna `realmente_no_pago` de arriba hacia abajo. Entre los clientes al día (atraso 0 o negativo) el
impago ronda el 13 a 17%, casi plano. Con **un** mes de atraso salta a cerca de 33%. Con **dos** meses salta a
cerca de 70%, y de ahí en adelante se queda ahí. Es una **escalera con dos peldaños**, no una rampa.

La logística lineal no puede dibujar una escalera: como $z$ es lineal en `atraso_sep`, está obligada a subir el
riesgo al mismo ritmo por cada mes de atraso, y se equivoca en las dos puntas: subestima el riesgo de los que
están al día con atraso negativo, sobreestima el de los que tienen atraso 0, y se queda corta en el peldaño de
2. La de grado 2 mejora porque un cuadrado puede curvar la rampa, pero sigue sin poder hacer un escalón. El
boosting reproduce cada peldaño casi al centavo, porque un árbol hace justamente eso: pregunta "¿atraso mayor o
igual a 2?" y asigna un riesgo distinto a cada lado. Veámoslo como curva, tomando un cliente típico y variando
solo el atraso de septiembre.


In [ ]:

tipico = X_tr.median()
grilla = pd.DataFrame([tipico] * 11); grilla["atraso_sep"] = np.arange(-2, 9); grilla = grilla[X.columns]
G = esc.transform(grilla)
curva = pd.DataFrame({"atraso_sep": grilla["atraso_sep"],
                      "logística lineal": logit.predict_proba(G)[:, 1],
                      "logística grado 2": poli_cv.predict_proba(esc2.transform(poly.transform(G)))[:, 1],
                      "gradient boosting": gb.predict_proba(G)[:, 1]}).melt(id_vars="atraso_sep", var_name="modelo", value_name="riesgo estimado")
fig = px.line(curva, x="atraso_sep", y="riesgo estimado", color="modelo", markers=True, title="Riesgo estimado según meses de atraso, para un cliente típico")
fig.update_layout(height=400); fig.show()



### Qué variables usa cada uno

La logística tiene coeficientes; el boosting reporta la **importancia** de cada variable, que es cuánto
contribuyó cada una a reducir la impureza a lo largo de todos los árboles (una fracción que suma 1). No son la
misma cosa y no se pueden comparar en magnitud, pero sí en **orden**: si dos modelos de familias distintas
coinciden en qué variables van primero, la lista se puede llevar a una reunión.


In [ ]:

imp = pd.DataFrame({"coef. logística (abs)": np.abs(logit.coef_[0]), "importancia boosting": gb.feature_importances_, "importancia random forest": bosque.feature_importances_}, index=X.columns)
imp["ranking logística"] = imp["coef. logística (abs)"].rank(ascending=False).astype(int)
imp["ranking boosting"] = imp["importancia boosting"].rank(ascending=False).astype(int)
print(imp.sort_values("importancia boosting", ascending=False).head(10).round(3))



## 8. Usar el ganador: umbral con costos y calibración

Un AUC más alto no es una decisión. Para usar el modelo hay que fijar el umbral, y del notebook anterior
sabemos cómo: con los costos de cada error. Supongamos que dejar pasar a un cliente que no paga cuesta 4 veces
lo que rechazar a uno que sí pagaba ($c_{FN} = 4$, $c_{FP} = 1$). La fórmula dice umbral $= 1/5 = 0{,}2$ si
las probabilidades están calibradas; la tabla sobre testing dice dónde cae el mínimo de verdad. Y como aquí
tenemos 9.000 clientes de testing, las dos deberían parecerse más que en German Credit.

También revisamos la calibración por deciles: los ensambles de árboles no siempre entregan probabilidades bien
calibradas (el random forest, en particular, tiende a no acercarse nunca a 0 ni a 1 porque promedia), y si el
umbral se elige por fórmula eso importa.


In [ ]:

c_FN, c_FP = 4, 1
umbrales = np.abs(np.round(np.arange(1.0, -0.001, -0.05), 2))
def costo_por_umbral(p):
    filas = []
    for u in umbrales:
        pred = (p > u).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
        filas.append(dict(umbral=u, costo=fp * c_FP + fn * c_FN, recall=tp / (tp + fn), precision=tp / (tp + fp) if tp + fp else 0))
    return pd.DataFrame(filas)

print("Umbral por fórmula:", round(c_FP / (c_FP + c_FN), 3))
for nombre in ["logística lineal", "logística grado 2, L2 por validación cruzada", "random forest, 300 árboles", "gradient boosting, 300 árboles"]:
    t = costo_por_umbral(RESULTADOS[nombre]["p_te"]); f = t.loc[t.costo.idxmin()]
    print(f"{nombre:45s} umbral óptimo (tabla) {f.umbral:.2f}   costo mínimo {int(f.costo)}   recall {f.recall:.3f}  precision {f.precision:.3f}   (costo con umbral 0,5: {int(t.loc[t.umbral == 0.5, 'costo'].iloc[0])})")


In [ ]:

cal = pd.DataFrame({"y": y_te, "logística": RESULTADOS["logística lineal"]["p_te"], "random forest": RESULTADOS["random forest, 300 árboles"]["p_te"], "boosting": RESULTADOS["gradient boosting, 300 árboles"]["p_te"]})
out = {}
for m in ("logística", "random forest", "boosting"):
    g = pd.qcut(cal[m], 10, labels=False, duplicates="drop")
    out[m + ": p promedio"] = cal.groupby(g)[m].mean().values
    out[m + ": real"] = cal.groupby(g)["y"].mean().values
print("Calibración por deciles de riesgo (p promedio del decil contra fracción real que no pagó):")
print(pd.DataFrame(out).round(3).to_string())



Lectura de la calibración: el boosting y el random forest están bien calibrados en todos los deciles (aquí el
random forest no tuvo el problema de encogerse hacia el centro, porque con 21.000 clientes y hojas de 5 sus
árboles llegan lejos). La peor calibrada es la logística lineal: en los deciles medios asigna entre 0,18 y 0,23
donde la realidad es 0,12 a 0,15, y en el decil más alto asigna 0,59 donde la realidad es 0,67. Es la misma
escalera de la sección 7 vista desde otro lado: la recta reparte el riesgo en una rampa y la realidad lo
concentra en un escalón. Por eso su umbral de tabla (0,25) se aleja de la fórmula (0,20) y el del boosting no.

El modelo que se va a usar es el boosting, con el umbral que salió de la tabla de costos, y con la calibración
revisada. En el negocio eso se traduce en cuántos clientes rechaza y cuántos impagos evita, que es lo
que la fila de recall y precision con ese umbral dice. Un ejercicio abajo pide medir cuánto se ahorra respecto
de la logística en pesos de costo.



## 9. Lo que hay que llevarse

1. La regularización importa cuando el modelo tiene muchos pesos **respecto de las filas**: con 299 columnas de
   grado 2 y 21.000 clientes casi no cambia nada; con las mismas 299 columnas y 1.000 clientes es lo que separa
   un modelo que generaliza de uno que memoriza. En German Credit, con 18 columnas y 700 filas, no había nada que
   frenar.
2. Un **árbol** es una secuencia de preguntas; dibuja escalones y memoriza si se le deja crecer. Un solo árbol
   podado no le gana a una buena logística.
3. Un **ensamble** convierte muchos árboles en un modelo bueno: random forest promedia árboles independientes
   y profundos (casi imposible de arruinar); gradient boosting suma árboles chicos que se corrigen en secuencia
   (el más preciso con datos tabulares, y el único de los dos que puede sobreajustar con demasiadas rondas).
4. **Comparar** exige intervalos: bootstrap para cada modelo, diferencia pareada entre dos, validación cruzada
   para ver si se sostiene. Con 9.000 clientes de testing, centésimas de AUC son reales; con 300, no.
5. Cuando un modelo gana, hay que **mirar qué encontró**. Aquí fue una escalera en los meses de atraso, y esa
   forma explica por qué una recta pierde, un polinomio empata a medias y un árbol acierta.
6. Un AUC no es una decisión: el modelo se usa con un **umbral elegido con costos**, y con la calibración
   revisada, porque los ensambles no siempre entregan probabilidades honestas.



## Ejercicios

**Ejercicio 1.** Entrena el gradient boosting con 30, 100, 300 y 1.000 árboles (misma tasa 0,05). Reporta AUC de
training y de testing para cada uno. ¿Mejora siempre? ¿Dónde empieza a abrirse la brecha? Compara con el random
forest con los mismos cuatro tamaños: ¿le pasa lo mismo?

**Ejercicio 2.** Repite la tabla de la sección 7 para `edad` (agrúpala en tramos de 5 años). ¿Hay una forma no
lineal ahí también, o es una rampa que la logística ya dibuja bien?

**Ejercicio 3.** Con $c_{FN} = 4$ y $c_{FP} = 1$, calcula el costo total en testing del boosting y de la
logística lineal, cada uno con su umbral óptimo de la tabla. ¿Cuántas unidades de costo se ahorran con el
boosting sobre 9.000 clientes? Ponle un intervalo por bootstrap pareado a ese ahorro.

**Ejercicio 4.** Entrena el boosting solo con los primeros 1.000 clientes de training. Mide AUC de training y
de testing. ¿Le sigue ganando a la logística lineal entrenada con los mismos 1.000? Antes de correrlo, escribe
qué esperas, y después compara con lo que pasó en German Credit, donde con 1.000 clientes nada le ganaba a la
recta.



## Soluciones


In [ ]:

# SOLUCIÓN 1: número de árboles
print("gradient boosting (tasa 0,05, profundidad 3):")
for n in [30, 100, 300, 1000]:
    m = GradientBoostingClassifier(n_estimators=n, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=0).fit(A_tr, y_tr)
    print(f"   {n:5d} árboles  AUC train {roc_auc_score(y_tr, m.predict_proba(A_tr)[:, 1]):.3f}  test {roc_auc_score(y_te, m.predict_proba(A_te)[:, 1]):.3f}")
print("random forest:")
for n in [30, 100, 300, 1000]:
    m = RandomForestClassifier(n_estimators=n, min_samples_leaf=5, n_jobs=-1, random_state=0).fit(A_tr, y_tr)
    print(f"   {n:5d} árboles  AUC train {roc_auc_score(y_tr, m.predict_proba(A_tr)[:, 1]):.3f}  test {roc_auc_score(y_te, m.predict_proba(A_te)[:, 1]):.3f}")
print("Lectura: en el boosting el AUC de testing sube y después se estanca o baja mientras el de training sigue subiendo (la brecha se abre); en el random forest agregar árboles solo estabiliza.")


In [ ]:

# SOLUCIÓN 2: edad
comp["tramo_edad"] = (X_te["edad"].values // 5) * 5
por_edad = comp.groupby("tramo_edad").agg(clientes=("realmente_no_pago", "size"), realmente_no_pago=("realmente_no_pago", "mean"), logistica=("logistica", "mean"), boosting=("boosting", "mean"))
print(por_edad[por_edad.clientes >= 50].round(3))
print("Lectura: la edad es casi plana o una rampa suave; la logística y el boosting le asignan casi lo mismo. La no linealidad que importa está en los atrasos, no aquí.")


In [ ]:

# SOLUCIÓN 3: ahorro en costo con el umbral óptimo de cada uno
def costo_min(p, idx=None):
    yy = y_te if idx is None else y_te[idx]; pp = p if idx is None else p[idx]
    mejor = None
    for u in umbrales:
        pred = (pp > u).astype(int); tn, fp, fn, tp = confusion_matrix(yy, pred, labels=[0, 1]).ravel()
        c = fp * c_FP + fn * c_FN
        if mejor is None or c < mejor: mejor = c
    return mejor
pb, pl = RESULTADOS["gradient boosting, 300 árboles"]["p_te"], RESULTADOS["logística lineal"]["p_te"]
ahorro = costo_min(pl) - costo_min(pb)
d = np.array([costo_min(pl, i) - costo_min(pb, i) for i in IDX_BS[:200]])
print(f"Costo mínimo logística: {costo_min(pl)}   boosting: {costo_min(pb)}   ahorro: {ahorro} unidades sobre 9.000 clientes   IC 95% pareado [{np.percentile(d, 2.5):.0f}, {np.percentile(d, 97.5):.0f}]")


In [ ]:

# SOLUCIÓN 4: con solo 1.000 clientes de training
A_chico, y_chico = A_tr[:1000], y_tr[:1000]
gb_chico = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=0).fit(A_chico, y_chico)
lg_chico = LogisticRegression(penalty=None, max_iter=5000).fit(A_chico, y_chico)
for nombre, m in [("boosting con 1.000", gb_chico), ("logística con 1.000", lg_chico)]:
    print(f"{nombre:22s} AUC train {roc_auc_score(y_chico, m.predict_proba(A_chico)[:, 1]):.3f}  test {roc_auc_score(y_te, m.predict_proba(A_te)[:, 1]):.3f}")
print("Lectura: con 1.000 clientes el boosting memoriza (AUC de training cerca de 1) y su AUC de testing cae de 0,78 a 0,75, pero sigue ganándole a la logística por unos 0,05. La diferencia con German Credit no es solo el tamaño: aquí existe una escalera fuerte y simple en los atrasos que hasta un boosting con pocos datos encuentra, y allá no había nada no lineal que encontrar. El ensamble gana cuando hay algo que descubrir; los datos abundantes lo que hacen es cerrar la brecha entre training y testing.")
